<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 07 — Support Vector Machine (SVM)
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Margens, Kernels e Quarto Modelo no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">⚔️ 4º Modelo do Curso</span>
</div>


## Onde estamos?

Nos últimos notebooks treinamos três modelos no Titanic:

| Aula | Modelo | Ideia central |
|------|--------|---------------|
| 04 | Regressão Linear | Prever um número via reta de mínimos quadrados |
| 05 | Regressão Logística | Probabilidade via sigmoide |
| 06 | KNN | Voto dos K vizinhos mais próximos |
| **07** | **SVM** | **Hiperplano de margem máxima** |

Hoje aprendemos o **SVM (Support Vector Machine)**. Enquanto o KNN pergunta
*"quem são os mais parecidos?"* e a Regressão Logística pergunta *"qual a
probabilidade?"*, o SVM faz uma pergunta diferente:

> *"Qual é a fronteira que separa as classes com a maior distância possível?"*

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Hiperplano de margem máxima — vetores de suporte |
| **2** | O parâmetro C — margem rígida vs suave |
| **3** | Kernel Trick e aplicação no Titanic |
| **4** | Ajustando hiperparâmetros com Grid Search |
| **5** | Placar final — KNN vs Reg. Logística vs SVM |

> **Tempo estimado: 35 minutos**

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#0c5460;">Destaque desta aula:</strong>
<span style="color:#0c5460;"> o Kernel Trick (Parte 3) é um dos truques matemáticos mais elegantes do ML — permite ao SVM aprender fronteiras curvas sem transformar explicitamente os dados.</span>
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# 5 features simples e fáceis de explicar
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição — O Hiperplano de Margem Máxima</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Não basta separar as classes — queremos separar com a maior folga possível."</p>
    </div>
</div>

### O problema fundamental da classificação

Imagine dois grupos de pontos num plano. Existem infinitas retas que os
separam. Qual é a **melhor**?

O SVM responde: a reta (ou **hiperplano** em N dimensões) que **maximiza a
distância até os pontos mais próximos de cada classe**. Essa distância é
chamada de **margem**.

### Por que maximizar a margem?

Uma margem maior significa **mais tolerância a erros em dados novos**. É como
estacionar numa vaga folgada vs apertada: na folgada, errar 5cm não causa
problema; na apertada, causa.

### Vocabulário essencial

| Termo | Significado |
|-------|-------------|
| **Hiperplano** | A fronteira de decisão — reta em 2D, plano em 3D |
| **Vetores de Suporte** | Os pontos **mais próximos** do hiperplano — são eles que definem a margem |
| **Margem** | Distância entre o hiperplano e os vetores de suporte de cada classe |

<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:12px 18px; border-radius:6px; margin:12px 0;">
<strong style="color:#5b2c8d;">Por que "Support Vector Machine"?</strong>
<span style="color:#5b2c8d;"> Se você remover qualquer ponto do dataset exceto os vetores de suporte e retreinar, o hiperplano permanece exatamente o mesmo. Só esses pontos importam para definir a fronteira — daí o nome.</span>
</div>


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Observe o diagrama abaixo e responda antes de rodar o código: (a) por que a Reta B é melhor que a Reta A mesmo que ambas separem as classes? (b) o que acontece se um novo ponto aparecer perto da fronteira?</span></div>

*✏️ (a) A Reta B é melhor porque: `???`*

*✏️ (b) Para novos pontos próximos à fronteira, a Reta `???` é mais robusta porque: `???`*


In [ ]:
# Visualizando o conceito de margem máxima
np.random.seed(7)

X_neg = np.random.randn(20, 2) + np.array([-2.0, -1.0])
X_pos = np.random.randn(20, 2) + np.array([ 2.0,  1.0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Por que Maximizar a Margem?", fontsize=13, fontweight="bold")

x_line = np.linspace(-5, 5, 200)

for ax, titulo, retas in [
    (axes[0], "Muitas retas separam as classes\nQual escolher?",
     [(-0.3, 0.2,  "#aaa", "--", "Reta A (funciona)"),
      ( 0.5, 0.1,  "#888", "-.", "Reta B (funciona)"),
      ( 0.0, 0.5,  "#bbb", ":",  "Reta C (funciona)")]),
    (axes[1], "SVM escolhe a de MARGEM MÁXIMA",
     [( 0.5, 0.0, "#0f3460", "-",  "Hiperplano ótimo"),
      ( 0.5, 1.2, "#0f3460", "--", "Margem superior"),
      ( 0.5,-1.2, "#0f3460", "--", "Margem inferior")])
]:
    ax.scatter(X_neg[:,0], X_neg[:,1], c="#e94560", s=60, edgecolors="white",
               linewidth=0.8, label="Classe 0", zorder=5)
    ax.scatter(X_pos[:,0], X_pos[:,1], c="#0f3460", s=60, edgecolors="white",
               linewidth=0.8, label="Classe 1", zorder=5)

    for b0, b1, cor, ls, lbl in retas:
        ax.plot(x_line, b0*x_line + b1, color=cor, linestyle=ls,
                linewidth=2, label=lbl, alpha=0.85)

    if titulo.startswith("SVM"):
        ax.fill_between(x_line, 0.5*x_line - 1.2, 0.5*x_line + 1.2,
                         alpha=0.06, color="#0f3460", label="Zona da margem")
        sv_neg = X_neg[np.argmin(X_neg[:,0])]
        sv_pos = X_pos[np.argmin(X_pos[:,0])]
        for sv, cor in [(sv_neg,"#e94560"),(sv_pos,"#0f3460")]:
            ax.scatter(*sv, s=180, facecolors="none", edgecolors=cor,
                       linewidth=2.5, zorder=10)

    ax.set_xlim(-5.5, 5.5); ax.set_ylim(-5, 5)
    ax.set_title(titulo, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("(a) A Reta B (hiperplano de margem máxima) é melhor porque:")
# print("    Ela está equidistante dos pontos mais próximos de cada classe.")
# print("    As outras retas ficam muito perto de um dos grupos, deixando")
# print("    pouca 'folga' para erros em dados novos.")
# print()
# print("(b) Para novos pontos próximos à fronteira:")
# print("    A reta de margem máxima é mais robusta porque tem o maior espaço")
# print("    de segurança. Retas com margem pequena erram com qualquer")
# print("    variação mínima (ruído, medição imprecisa).")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Margem Rígida vs Suave — O Parâmetro C</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Na vida real, os dados têm ruído. O SVM precisa ser tolerante."</p>
    </div>
</div>

### O problema da margem rígida

A ideia da Parte 1 é chamada de **margem rígida**: nenhum ponto pode cruzar a
margem. Na prática, dados reais têm ruído e sobreposição entre classes — a
margem rígida falha nesses casos.

### A solução: margem suave

O SVM com margem suave permite que **alguns pontos violem a margem**, mas
cobra um custo por isso. O **parâmetro C** controla esse trade-off:

```
C grande  →  punição alta por violações  →  margem menor, tenta acertar tudo
             (mais rígido → risco de overfitting)

C pequeno →  punição baixa por violações →  margem maior, aceita mais erros
             (mais tolerante → melhor generalização)
```

<div style="background:#fff3cd; border-left:5px solid #856404; padding:12px 18px; border-radius:6px; margin:12px 0;">
<strong style="color:#856404;">C é o hiperparâmetro mais importante do SVM.</strong>
<span style="color:#856404;"> O padrão do scikit-learn é C=1.0, um bom ponto de partida.</span>
</div>


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_classification

np.random.seed(42)

# Dataset 2D com alguma sobreposição entre classes + outliers
X_2d, y_2d = make_classification(
    n_samples=150, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    class_sep=0.7, random_state=42
)
X_2d = np.vstack([X_2d, [[-0.5, 2.5], [0.8, -2.5]]])
y_2d = np.concatenate([y_2d, [1, 0]])

X_2d_tr, X_2d_te, y_2d_tr, y_2d_te = train_test_split(
    X_2d, y_2d, test_size=0.3, random_state=42)

# Testando diferentes valores de C
valores_C = [0.01, 1.0, 100.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle("Efeito do Parâmetro C — Margem Rígida vs Suave", fontsize=13, fontweight="bold")

h = 0.05
for ax, C_val in zip(axes, valores_C):
    svm_c = SVC(kernel="linear", C=C_val, random_state=42)
    svm_c.fit(X_2d_tr, y_2d_tr)

    x_min, x_max = X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5
    y_min, y_max = X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
    Z = svm_c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.20, cmap="RdBu")

    ax.scatter(X_2d_tr[:,0], X_2d_tr[:,1], c=y_2d_tr,
               cmap="RdBu", edgecolors="white", s=40, linewidth=0.6, zorder=5)
    ax.scatter(svm_c.support_vectors_[:,0], svm_c.support_vectors_[:,1],
               s=180, facecolors="none", edgecolors="#f0a500",
               linewidth=2, zorder=10, label=f"SVs: {len(svm_c.support_vectors_)}")

    acc = svm_c.score(X_2d_te, y_2d_te)
    titulo = "Underfitting" if C_val < 0.1 else ("Overfitting" if C_val > 50 else "Equilibrado")
    ax.set_title(f"C = {C_val}\nAcurácia teste: {acc:.0%} ({titulo})", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlabel("F1"); ax.set_ylabel("F2")

plt.tight_layout()
plt.show()

print("Observe o número de vetores de suporte (SVs) em cada gráfico:")
print("  C pequeno → mais SVs → margem mais larga → mais tolerante")
print("  C grande  → menos SVs → margem mais estreita → mais rígido")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Olhando os 3 gráficos: (a) com C=0.01, o modelo está overfittando ou underfittando? (b) com C=100, o que acontece com a margem e os vetores de suporte? (c) qual C você escolheria como ponto de partida?</span></div>

*✏️ (a) C=0.01: o modelo está `???` porque: `???`*

*✏️ (b) C=100: margem `???`, vetores de suporte `???`*

*✏️ (c) Escolheria C=`???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("(a) C=0.01 — UNDERFITTING: margem muito larga, aceita muitas")
# print("    violações. O modelo ignora detalhes importantes dos dados.")
# print()
# print("(b) C=100 — margem ESTREITA, poucos vetores de suporte:")
# print("    a fronteira se dobra para tentar acertar todos os pontos,")
# print("    incluindo outliers. Risco de overfitting em dados novos.")
# print()
# print("(c) Ponto de partida recomendado: C=1.0 (padrão do sklearn) —")
# print("    equilibra rigidez e tolerância. Ajustamos depois com Grid Search.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Kernel Trick e o SVM no Titanic</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Quando uma reta não resolve, o kernel encontra outra forma de separar."</p>
    </div>
</div>

### O problema da não-linearidade

Muitos problemas reais não podem ser separados por uma reta. O exemplo
clássico: dois grupos concêntricos (um anel em torno do outro). Nenhuma reta
separa isso — mas um **círculo** separaria.

A sacada do SVM é o **Kernel Trick**: em vez de transformar os dados
explicitamente para uma dimensão maior, o kernel calcula diretamente o efeito
dessa transformação, sem precisar ir até lá.

| Kernel | Fronteira que aprende | Quando usar |
|--------|----------------------|-------------|
| **linear** | Reta / hiperplano | Dados linearmente separáveis |
| **rbf** | Curva flexível | Padrão — funciona na maioria dos problemas |

**Regra prática:** comece com `kernel='rbf'`, `C=1.0` e `gamma='scale'`.


In [ ]:
from sklearn.datasets import make_circles

np.random.seed(42)

# Dataset clássico onde nenhuma reta funciona: círculos concêntricos
X_c, y_c = make_circles(n_samples=150, noise=0.07, factor=0.4, random_state=42)
X_c_sc = StandardScaler().fit_transform(X_c)
X_c_tr, X_c_te, y_c_tr, y_c_te = train_test_split(X_c_sc, y_c, test_size=0.25, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle("Kernel Trick — Linear vs RBF em Dados Não-Lineares", fontsize=13, fontweight="bold")

h = 0.04
for ax, kernel_nome, params in [(axes[0], "linear", {"C":1.0}),
                                  (axes[1], "rbf",    {"C":1.0, "gamma":"scale"})]:
    svm_k = SVC(kernel=kernel_nome, **params, random_state=42)
    svm_k.fit(X_c_tr, y_c_tr)
    acc = svm_k.score(X_c_te, y_c_te)

    x_min, x_max = X_c_sc[:,0].min()-0.3, X_c_sc[:,0].max()+0.3
    y_min, y_max = X_c_sc[:,1].min()-0.3, X_c_sc[:,1].max()+0.3
    xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
    Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdBu")
    ax.scatter(X_c_sc[:,0], X_c_sc[:,1], c=y_c, cmap="RdBu",
               edgecolors="white", s=35, linewidth=0.5, zorder=5)

    titulo_col = "SVM Linear" if kernel_nome == "linear" else "SVM RBF"
    ok = "✅" if acc > 0.85 else ("⚠️" if acc > 0.7 else "❌")
    ax.set_title(f"{titulo_col}\nAcurácia: {acc:.0%} {ok}", fontweight="bold")
    ax.set_xlabel("F1"); ax.set_ylabel("F2")

plt.tight_layout()
plt.show()

print("O kernel RBF consegue aprender a fronteira circular.")
print("O kernel linear falha — não existe reta que separe círculos concêntricos.")


### Aplicando ao Titanic

Agora que entendemos margem e kernel, aplicamos o SVM ao problema real:
prever se um passageiro sobreviveu. Testamos os dois kernels principais.

Note o `probability=True` — sem isso, o SVM não produz probabilidades e não
conseguiríamos calcular AUC-ROC depois. Isso tem um pequeno custo computacional.


In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay)

svm_linear = SVC(kernel="linear", C=1.0, probability=True, random_state=42)
svm_linear.fit(X_treino_sc, y_treino)

svm_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
svm_rbf.fit(X_treino_sc, y_treino)

y_pred_lin = svm_linear.predict(X_teste_sc)
y_pred_rbf = svm_rbf.predict(X_teste_sc)

print("SVM no Titanic — comparação Linear vs RBF")
print(f"  {'Métrica':<12} {'Linear':>10} {'RBF':>10}")
for nome, func in [("Acurácia", accuracy_score), ("F1-Score", f1_score)]:
    print(f"  {nome:<12} {func(y_teste, y_pred_lin):>10.4f} {func(y_teste, y_pred_rbf):>10.4f}")


In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle("SVM no Titanic — Matrizes de Confusão", fontweight="bold")

for ax, y_pred, kernel_nome, cmap in [(axes[0], y_pred_lin, "Linear", "Blues"),
                                        (axes[1], y_pred_rbf, "RBF", "Purples")]:
    acc = accuracy_score(y_teste, y_pred)
    cm  = confusion_matrix(y_teste, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Não Sobrev.","Sobreviveu"]).plot(
        ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f"SVM {kernel_nome}\nAcurácia: {acc:.1%}", fontweight="bold")

plt.tight_layout()
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Compare os dois SVMs: (a) qual teve maior acurácia? (b) para o contexto do Titanic, qual dos dois você escolheria e por quê?</span></div>

*✏️ (a) Maior acurácia: SVM `???`*

*✏️ (b) Escolheria: `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print("O Titanic tem uma fronteira parcialmente não-linear (interações entre")
# print("classe social, gênero e idade). O SVM RBF costuma capturar melhor")
# print("essas interações, embora a diferença nem sempre seja grande.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Ajustando Hiperparâmetros com Grid Search</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Nenhum hiperparâmetro é bom por padrão — precisamos testar."</p>
    </div>
</div>

### O que é validação cruzada?

Até agora, avaliamos o modelo só com **um** split treino/teste. Mas e se esse
split específico tiver sido "sortudo" ou "azarado"? A **validação cruzada**
(cross-validation) resolve isso: divide o treino em vários pedaços (chamados
"folds"), treina e testa várias vezes trocando qual pedaço é usado como teste,
e tira a média. Isso dá uma estimativa mais confiável do desempenho.

### O que é Grid Search?

Em vez de testar um valor de C de cada vez, o **GridSearchCV** testa **todas
as combinações** de valores que você especificar, usando validação cruzada em
cada uma:

```
grade = {"C": [0.1, 1.0, 10.0], "gamma": ["scale", 0.01, 0.1]}
→ 3 × 3 = 9 combinações × 5 folds = 45 treinos no total
```

**Por que não usar o conjunto de teste para isso?** Ele deve permanecer
intocado até a avaliação final — se ajustarmos hiperparâmetros olhando o
teste, "contaminamos" a avaliação. O Grid Search usa só os dados de treino.


In [ ]:
from sklearn.model_selection import GridSearchCV

grade = {
    "C":     [0.1, 1.0, 10.0],
    "gamma": ["scale", 0.01, 0.1],
}

grid_svm = GridSearchCV(
    SVC(kernel="rbf", probability=True, random_state=42),
    grade, cv=5, scoring="f1", n_jobs=-1,
)
grid_svm.fit(X_treino_sc, y_treino)

print("GRID SEARCH — SVM RBF")
print(f"Melhores hiperparâmetros: {grid_svm.best_params_}")
print(f"Melhor F1 (validação cruzada 5-fold): {grid_svm.best_score_:.4f}")


In [ ]:
# Heatmap de Grid Search — visualizando o espaço de hiperparâmetros
resultados = pd.DataFrame(grid_svm.cv_results_)
resultados_pivot = resultados.pivot_table(
    index="param_C", columns="param_gamma", values="mean_test_score")

plt.figure(figsize=(7, 4.5))
sns.heatmap(resultados_pivot, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.5, annot_kws={"size": 10})
plt.title("Grid Search — F1 por C e gamma\n(mais escuro = melhor combinação)", fontweight="bold")
plt.xlabel("gamma"); plt.ylabel("C")
plt.tight_layout()
plt.show()

melhor_C     = grid_svm.best_params_["C"]
melhor_gamma = grid_svm.best_params_["gamma"]
print(f"Melhor combinação: C={melhor_C}, gamma={melhor_gamma}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Interprete o heatmap: (a) em qual região (C alto ou baixo, gamma alto ou baixo) o modelo performa melhor? (b) o Grid Search melhorou o F1 em relação ao SVM padrão (C=1, gamma=scale) que treinamos na Parte 3?</span></div>

*✏️ (a) Melhor região: C `???` e gamma `???`*

*✏️ (b) Melhoria do Grid Search em relação ao padrão: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# svm_otimizado = grid_svm.best_estimator_
# y_pred_otim   = svm_otimizado.predict(X_teste_sc)
#
# f1_padrao = f1_score(y_teste, y_pred_rbf)
# f1_otim   = f1_score(y_teste, y_pred_otim)
#
# print(f"SVM RBF padrão (C=1, gamma=scale): F1 = {f1_padrao:.4f}")
# print(f"SVM RBF otimizado (Grid Search):    F1 = {f1_otim:.4f}")
# print(f"Ganho: {(f1_otim - f1_padrao)*100:+.2f} pontos percentuais")
# print()
# print("Em datasets pequenos como este, o ganho costuma ser modesto.")
# print("Em datasets grandes e complexos, o Grid Search pode ser decisivo.")


In [ ]:
# Guardando o modelo otimizado para a comparação final
svm_otimizado = grid_svm.best_estimator_
y_pred_otim   = svm_otimizado.predict(X_teste_sc)
print(f"SVM otimizado pronto: C={melhor_C}, gamma={melhor_gamma}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Placar Final — KNN vs Reg. Logística vs SVM</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Três modelos, um dataset, métricas iguais — quem vence no Titanic?"</p>
    </div>
</div>

Chegamos ao fim da trilogia de modelos clássicos. Vamos treinar rapidamente o
KNN e a Regressão Logística (das Aulas 05 e 06) para comparar os três, lado a
lado, nos mesmos dados de teste.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve

knn_ref = KNeighborsClassifier(n_neighbors=7).fit(X_treino_sc, y_treino)
lr_ref  = LogisticRegression(max_iter=1000, random_state=42).fit(X_treino_sc, y_treino)

modelos = {
    "KNN (K=7)":       knn_ref,
    "Reg. Logística":  lr_ref,
    "SVM (otimizado)": svm_otimizado,
}

tabela = []
for nome, modelo in modelos.items():
    yp  = modelo.predict(X_teste_sc)
    ypr = modelo.predict_proba(X_teste_sc)[:, 1]
    tabela.append({
        "Modelo":   nome,
        "Acurácia": accuracy_score(y_teste, yp),
        "Precisão": precision_score(y_teste, yp),
        "Recall":   recall_score(y_teste, yp),
        "F1":       f1_score(y_teste, yp),
        "AUC-ROC":  roc_auc_score(y_teste, ypr),
    })

df_tabela = pd.DataFrame(tabela).set_index("Modelo")
print("PLACAR GERAL — Todos os Modelos")
print(df_tabela.round(4).to_string())


In [ ]:
# Curvas ROC sobrepostas
plt.figure(figsize=(7, 6))
cores = ["#a8d8ea", "#0f3460", "#e94560"]
for cor, (nome, modelo) in zip(cores, modelos.items()):
    ypr = modelo.predict_proba(X_teste_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_teste, ypr)
    auc = roc_auc_score(y_teste, ypr)
    plt.plot(fpr, tpr, color=cor, linewidth=2.2, label=f"{nome} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], "k--", linewidth=1, alpha=0.4, label="Aleatório")
plt.xlabel("Taxa de Falso Positivo")
plt.ylabel("Taxa de Verdadeiro Positivo (Recall)")
plt.title("Curvas ROC — Comparação Final", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 (final) — Analisando a tabela e as curvas ROC: (a) qual modelo você colocaria em produção para o Titanic e por quê? (b) para um projeto em que a velocidade de predição importa muito (milhões de previsões por segundo), qual modelo você descartaria?</span></div>

*✏️ (a) Colocaria em produção: `???` porque: `???`*

*✏️ (b) Descartaria para predição rápida: `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print("(a) Não há resposta única — depende dos requisitos:")
# print("    Se INTERPRETABILIDADE importa: Regressão Logística (coeficientes claros)")
# print("    Se PERFORMANCE máxima importa: SVM otimizado (fronteiras não-lineares)")
# print()
# print("(b) Descartar para predição rápida: KNN.")
# print("    O KNN precisa calcular distâncias para TODOS os pontos de treino")
# print("    a cada nova previsão. Com milhões de amostras, isso é inviável.")
# print("    Regressão Logística e SVM só aplicam uma fórmula já pronta —")
# print("    muito mais rápidos na hora de prever.")


**✏️ Minha reflexão sobre a aula:**

1. Vetor de suporte é: *...* — tem esse nome porque: *...*

2. O Kernel Trick é elegante porque: *...*

3. Usaria SVM em vez de Regressão Logística quando: *...*
